<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Hyperparameters, GridSearch, and Pipelines

_Authors:_ Tim Book et al

---

![](https://snag.gy/aYcCt2.jpg)

### Learning Objectives
- Describe what the terms hyperparameters, GridSearch, and pipeline mean.
- Apply `sklearn`'s `GridSearchCV` object.
- Use attributes of the GridSearch object.
- Describe the pitfalls of searching large hyperparameter spaces.
- Build pipelines.

In [ ]:
# Imports!
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [ ]:
# Column names.
abalone_colnames = [
    'sex', 'length', 'diameter', 'height', 'whole_weight',
    'shucked_weight', 'viscera_weight', 'shell_weight', 'rings'
]

# Pass in column names on import.


In [ ]:
# How many rows/columns?


In [ ]:
# Any missing values?


### Let's Whiz Through the Boring Parts
Quickly so as to get to the fun parts, I'll:
* Create our `X` and `y`. We'll be doing a kNN, so we won't use `sex`.
* Do a train-test split.
* Scale our data.

In [ ]:
# Define y.
y = ab['rings']

# Define X without rings or sex.
X = ab.drop(columns=['sex', 'rings'])

# Set up a train-test split.
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2022)

# Instantiate and fit StandardScaler, then transform our training and testing data.
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

# GridSearch
> _Oftentimes, we'll prefer RandomizedSearch. But, GridSearch is a good place to start!_

Up until now, we've tuned our hyperparameters (also called "tuning parameters") either automatically using some `sklearn` tech, or via sloppy loops. `sklearn` gives us some fun tools for automating this process in many cases. I'll demonstrate with a simple kNN first.

In [ ]:
# Instantiate a k-nearest neighbors model for regression.


# Set up our grid of hyperparameters.


# Instantiate our GridSearchCV object.


In [ ]:
# Let's cross-validate over some values of k.

In [ ]:
# What is our best value of k?

# How is that measured?

In [ ]:
# Let's retrieve our best estimator.

In [ ]:
# Let's look at our cross-validated results!

In [ ]:
# Let's plot these results.


This is cool!!! And we actually have `gs.best_estimator_` already fit as our optional model. But there's actually one more piece of efficiency `sklearn` can give us:

# Pipelines
Pipelines in `sklearn` allow you to combine several steps into one `estimator` or `transformer` object. 

That is, you can have pipelines of two forms:
1. one or more transformers (note this is the same as using a `ColumnTransformer` on all columns)
2. one or more transformers, with one estimator as the final stage

In [ ]:
# Instantiate a Pipeline which will scale our columns and then build a kNN model with 31 neighbors.

In [ ]:
# After instantiating, let's fit our Pipeline to our training data.

In [ ]:
# Now, let's pass our testing data through our already-fit Pipeline.

# Pipelines + GridSearch - All together, now!
You can, of course, use these all in conjunction. Let's try a new example with different data.

We'll use data on heart disease from a studen on men in Western Cape, South Africa. You can find a better description of the data [here](https://hastie.su.domains/ElemStatLearn/datasets/SAheart.info.txt).

In [ ]:
# Read in our heart data. What do you notice about famhist?
heart = pd.read_csv('./data/SAheart.data')
heart.head()

In [ ]:
# Here's a new way to convert a variable to 1s and 0s.
# Generate the series of Trues/Falses based on the condition.
# Multiplied by 1 will convert True --> 1 and False --> 0.
 

# See our results.


# This is really only good for generating a single dummy variable.

In [ ]:
# Define X and y.

In [ ]:
# Train/test split.


### Warning! You could accidentally make too many pairs of parameters and your model could take an extremely long time to fit! Start small.

In [ ]:
# Set up hyperparameters dictionary.


# Instantiate Pipeline to scale and build a logistic regression model.

# Instantiate our GridSearchCV object, so that we can get ready to fit this on our training data.


In [ ]:
%%time
# Let's time our work.

# Fit our GridSearchCV object to the training data.

In [ ]:
# Let's store our cross-validated results, so that we can use them later.

In [ ]:
# Let's plot these results.

plt.plot(
    df_gs.loc[df_gs.param_lr__penalty == 'l1', 'param_lr__C'],
    df_gs.loc[df_gs.param_lr__penalty == 'l1', 'mean_test_score'],
    c='navy',
    label='LASSO'
)
plt.plot(
    df_gs.loc[df_gs.param_lr__penalty == 'l2', 'param_lr__C'],
    df_gs.loc[df_gs.param_lr__penalty == 'l2', 'mean_test_score'],
    c='orange',
    label='Ridge'
)
plt.legend(loc=4);

In [ ]:
# What are the best parameters of our model?


## A Word of Caution on GridSearching

`sklearn` models often have many hyperparameters with many different possible values. It may be tempting to search over a wide variety of them. In general, this is not wise.

<details><summary>Why not?</summary>

- Remember that GridSearch searches over **all possible combinations of hyperparameters in the parameter dictionary!**

Imagine that we had this as our parameter dictionary:

```python
hyperparam_grid = {
    'n_neighbors': range(1, 151),
    'weights': ['uniform', 'distance', custom_function],
    'algorithm': ['ball_tree', 'kd_tree', 'brute', 'auto'],
    'leaf_size': range(1, 152),
    'metric': ['minkowski', 'euclidean'],
    'p': [1, 2]
}
```

**How many different combinations will need to be tested?**

| Hyperparameter | Number of Chosen Values |
| --- | --- |
| **n_neighbors** | 150 |
| **weights** | 3 |
| **algorithm** | 4 |
| **leaf_size** | 151 |
| **metric** | 2 |
| **p** | 2 |
| <br>_150 \* 3 \* 4 \* 151 \* 2 \* 2 = n combinations_ <br><br>| _1,087,200_ |

If we select `cv = 5`, we would fit 1,087,200 models on five folds, meaning we fit 5,436,000 models!

If you're not careful, GridSearching can quickly scale out of hand computationally.

> **It is extremely important to understand what the hyperparameters do and think critically about what ranges are useful and relevant to your model!**
</details>

## Alternative: RandomSearchCV
**Random searching** (aka **stochastic searching**) is a much more recommended method for large parameter spaces. It choses tuning parameter combinations at random to "visit" more of the parameter space without doing an exhaustive search.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
# RandomizedSearchCV()

## Interview Question

<details><summary>What is the difference between hyperparameters and statistical parameters?</summary>
    
- Statistical parameters are quantities that a model can learn or estimate. Examples include $\beta_0$ and $\beta_1$ in a linear model.
- Hyperparameters are quantities our model cannot learn, but affect the fit of our model. Examples include $k$ in $k$-nearest neighbors and $alpha$ in regularization.
</details>